In [1]:
import pandas as pd
from rouge_score import rouge_scorer
from sarathi.benchmark.request_generator.real_request_generator import RealRequestGenerator
from sarathi.benchmark.config import Config

/anaconda3/envs/vattn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


unable to import module pod_attn


2025-06-20 01:22:32,732	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
request_generator = RealRequestGenerator(config=Config({'num_requests': 100}))
cnn_prompts = request_generator.get_cnn_prompts()


In [3]:
scorer = rouge_scorer.RougeScorer(['rougeL'])

def get_rougeL_score(prompt, output):
    return scorer.score(prompt, output)['rougeL']

def get_avg_rougeL_score(prompts, outputs):
    precisions = []
    recalls = []
    fmeasures = []
    for prompt, output in zip(prompts, outputs):
        scores = get_rougeL_score(prompt, output)
        precisions.append(scores.precision)
        recalls.append(scores.recall)
        fmeasures.append(scores.fmeasure)
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls), sum(fmeasures) / len(fmeasures)


In [4]:
def get_rougeL_for_df(df, prompts):
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(prompts, df.output)
    return avg_rouge_score

In [5]:
df_rebatching = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b/req_100_batch_4_csv/rebatching.csv")
get_rougeL_for_df(df_rebatching, cnn_prompts)

0.20702352111421657

In [6]:
df_average = pd.read_csv("/workspace/xutingl/vattention-ee/outputs_13b/req_100_batch_4_csv/average.csv")
get_rougeL_for_df(df_average, cnn_prompts)

0.25474152566261593

In [23]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
precisions = []
recalls = []
fmeasures = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b/req_100_batch_1_csv/ee.csv")
avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
precisions.append(avg_precision)
recalls.append(avg_recall)
fmeasures.append(avg_fmeasure)

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    precisions.append(avg_precision)
    recalls.append(avg_recall)
    fmeasures.append(avg_fmeasure)
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "precision": precisions,
    "recall": recalls,
    "fmeasure": fmeasures
})
df

,policy,throughput,precision,recall,fmeasure
0,ee_nobatch,35.948304,0.345359,0.160167,0.193570
1,off,91.383235,0.444740,0.227738,0.281439
2,eager,100.560776,0.058651,0.064409,0.046470
3,lazy,89.529294,0.429197,0.223561,0.277634
4,average,95.754745,0.301047,0.182425,0.202879
5,rebatching,92.820732,0.219218,0.129791,0.139172


In [4]:
policies = ["off","eager", "average", "rebatching"]
throughputs = []
precisions = []
recalls = []
fmeasures = []


for policy in policies:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_13b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    precisions.append(avg_precision)
    recalls.append(avg_recall)
    fmeasures.append(avg_fmeasure)
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "precision": precisions,
    "recall": recalls,
    "fmeasure": fmeasures
})
df

,policy,throughput,precision,recall,fmeasure
0,off,99.306275,0.477297,0.222163,0.291897
1,eager,111.142936,0.096114,0.067653,0.070527
2,average,97.522551,0.452289,0.225127,0.288094
3,rebatching,107.896307,0.427697,0.225969,0.283217
